# Generators

> Generated from [`02-Generators.py`](02-Generators.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Generators: functions that use `yield` to produce values lazily, one at a time, instead of building a whole list upfront. Includes two-way communication (`send`), delegation (`yield from`), and generator-based pipelines.

### Key points
- A generator function uses `yield` instead of `return`. Calling it doesn't run the body — it returns a generator object; code runs (and pauses) as you call `next()` on it.
- Generator expressions: same syntax as a list comprehension but with `()` instead of `[]` — lazily evaluated, one value at a time.
- `StopIteration` is raised automatically once a generator's body finishes — same protocol as any iterator.
- `yield from other_gen` delegates to another generator/iterable, yielding all of its values in place.
- `gen.send(value)` sends a value INTO the generator at the point it's paused (received as the result of the `yield` expression). You must `next(gen)` once first to "prime" it (advance to the first `yield`).
- `gen.throw(Exception)` injects an exception at the current yield point; `gen.close()` stops the generator and runs any `finally` block.
- Infinite generators (`while True: yield ...`) are safe because nothing is computed until you actually pull values.
- Generators chain naturally into pipelines — each stage pulls from the previous one, nothing runs until the final consumer starts pulling (true lazy evaluation, useful for ETL-style filter -> transform -> aggregate).
- `next(gen, default)` returns `default` instead of raising `StopIteration` once exhausted — avoids a try/except for exhaustion checks.
- `@contextlib.contextmanager` is built on generators: code before `yield` acts as `__enter__`, code after acts as `__exit__` (full depth in the Context-Managers file).

### Gotchas (the part worth revising twice)
- You must call `next(gen)` once before the first `.send()` — sending into a generator that hasn't reached its first `yield` yet raises an error.
- A list comprehension materializes the whole thing in memory immediately; the generator-expression equivalent stays at a tiny, near-constant size no matter how large the range — this is the real reason to prefer generators for big data.
- Once a generator is exhausted, it stays exhausted — same one-shot behavior as iterators in general.
- `yield` pauses and resumes function state (locals, position) exactly the way `__enter__`/`__exit__` need to — that's *why* `@contextmanager` can be built from a plain generator function with a single `yield` splitting setup from teardown.

### When it matters
Use generators for large or streaming data (files, DB cursors, ETL pipelines), infinite sequences, or any time you want computation to happen only as values are actually consumed rather than all at once.

### One-line recap example
```python
def count_up_to(limit):
    n = 1
    while n <= limit:
        yield n
        n += 1

print(list(count_up_to(3)))  # [1, 2, 3] — computed lazily, one at a time
```

## 01. Generators - Introduction

- A generator is a special kind of iterator, created with `yield` instead
  of `return`, that produces values lazily (one at a time, on demand).
- Much less memory than building a full list upfront.

## 02. A Basic Generator Function

In [ ]:
def count_up_to(limit):
    n = 1
    while n <= limit:
        yield n  # pauses here, returns n, resumes on next call
        n += 1

print("\n# 02. A Basic Generator Function")
gen = count_up_to(5)
print("generator object:", gen)
print(next(gen))
print(next(gen))
for value in gen:  # continues from where next() left off
    print("remaining:", value)

## 03. Generators vs Regular Functions with Lists

In [ ]:
def squares_list(n):
    return [x ** 2 for x in range(n)]  # builds entire list in memory

def squares_gen(n):
    for x in range(n):
        yield x ** 2  # produces one value at a time

print("\n# 03. Generators vs Regular Functions with Lists")
print("list version:", squares_list(5))
print("generator version:", list(squares_gen(5)))
print("# - For huge n, squares_gen uses far less memory than squares_list.")

## 04. Generator Expressions

- Same syntax as list comprehension but with () - lazily evaluated.

In [ ]:
gen_expr = (x ** 2 for x in range(5))
print("\n# 04. Generator Expressions")
print(gen_expr)
print(list(gen_expr))

## 05. StopIteration Happens Automatically

In [ ]:
gen = count_up_to(2)
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration:
    print("StopIteration raised when generator is exhausted")

## 06. yield from - Delegating to Another Generator/Iterable

In [ ]:
def inner_gen():
    yield 1
    yield 2

def outer_gen():
    yield "start"
    yield from inner_gen()  # delegates - yields 1, then 2
    yield "end"

print("\n# 06. yield from")
print(list(outer_gen()))

## 07. send() - Sending Values Into a Generator

In [ ]:
def running_total():
    total = 0
    while True:
        value = yield total  # receives value sent via .send()
        total += value

print("\n# 07. send()")
gen = running_total()
next(gen)  # prime the generator (advance to first yield)
print(gen.send(10))
print(gen.send(5))
print(gen.send(20))

## 08. throw() and close() - Controlling a Generator

In [ ]:
def controllable_gen():
    try:
        while True:
            yield "running"
    except ValueError:
        yield "caught ValueError inside generator"
    finally:
        print("generator cleanup on close")

print("\n# 08. throw() and close()")
gen = controllable_gen()
print(next(gen))
print(gen.throw(ValueError))  # injects exception at the yield point
gen.close()  # stops the generator, runs finally block

## 09. Infinite Generators (safe because lazy)

In [ ]:
def natural_numbers():
    n = 1
    while True:
        yield n
        n += 1

print("\n# 09. Infinite Generators")
gen = natural_numbers()
first_five = [next(gen) for _ in range(5)]
print("first five naturals:", first_five)

## 10. Chaining Generators (pipeline style)

In [ ]:
def evens(iterable):
    for x in iterable:
        if x % 2 == 0:
            yield x

def doubled(iterable):
    for x in iterable:
        yield x * 2

print("\n# 10. Chaining Generators (pipeline style)")
pipeline = doubled(evens(range(10)))
print(list(pipeline))

## 11. Generator Memory Profiling vs List Comprehension

- A list comprehension materializes every value at once; a generator
  expression holds only its current state. For large ranges this is the
  difference between megabytes and a few dozen bytes.

In [ ]:
import sys

print("\n# 11. Generator Memory Profiling vs List Comprehension")
big_list_comp = [x * 2 for x in range(1_000_000)]
big_gen_expr = (x * 2 for x in range(1_000_000))
print("list comprehension size:", sys.getsizeof(big_list_comp), "bytes")
print("generator expression size:", sys.getsizeof(big_gen_expr), "bytes")

## 12. Coroutine-style Two-Way Communication - Running Average

- A more realistic use of send(): a generator that keeps a running average
  and yields stats back after every new value it receives.

In [ ]:
def running_average():
    count = 0
    total = 0
    average = None
    while True:
        value = yield {"count": count, "total": total, "average": average}
        count += 1
        total += value
        average = total / count

print("\n# 12. Coroutine-style Running Average")
stats_gen = running_average()
next(stats_gen)  # prime it
print(stats_gen.send(10))
print(stats_gen.send(20))
print(stats_gen.send(30))

## 13. contextlib.contextmanager - Built on Generators (reference)

- @contextlib.contextmanager turns a generator function into a context
  manager: code before `yield` is __enter__, code after is __exit__.
- Full depth (including exception handling inside it) lives in the
  Context-Managers file - this is just a note on WHY it works: a generator
  naturally pauses/resumes exactly like __enter__/__exit__ needs.

In [ ]:
import contextlib

@contextlib.contextmanager
def announce(name):
    print(f"entering {name}")   # this is the __enter__ half
    yield name
    print(f"exiting {name}")    # this is the __exit__ half

print("\n# 13. contextlib.contextmanager (built on generators)")
with announce("demo-block") as label:
    print("inside block:", label)

## 14. Generator Exhaustion Checks - next(gen, default)

- Instead of wrapping next() in try/except StopIteration, pass a default
  value as the second argument - it's returned once the generator is done.

In [ ]:
small_gen = count_up_to(2)
print(next(small_gen, "no more values"))
print(next(small_gen, "no more values"))
print(next(small_gen, "no more values"))  # exhausted -> sentinel instead of exception

## 15. Lazy ETL-style Pipeline - Filter -> Transform -> Aggregate

- Each stage is a generator that pulls from the previous one. Nothing runs
  until the final aggregate step starts consuming - true lazy evaluation.

In [ ]:
def extract(rows):
    yield from rows  # source data, could just as easily be a file or DB cursor

def filter_valid(rows):
    for row in rows:
        if row["amount"] > 0:  # drop bad/negative records
            yield row

def transform(rows):
    for row in rows:
        yield {**row, "amount_with_tax": round(row["amount"] * 1.1, 2)}

def aggregate_total(rows):
    total = 0
    for row in rows:
        total += row["amount_with_tax"]
    return total

print("\n# 15. Lazy ETL Pipeline (filter -> transform -> aggregate)")
raw_rows = [{"amount": 100}, {"amount": -5}, {"amount": 50}, {"amount": 25}]
etl_pipeline = transform(filter_valid(extract(raw_rows)))
print("final total after filter+transform+aggregate:", aggregate_total(etl_pipeline))